# 03 — Consolidación del último mes (SEPA diario → semestral)

Convierte el SEPA **diario** al formato **semestral wide** que leen los notebooks 01 y 02, para analizar el mes en curso sin esperar la consolidación oficial.

## Cómo usar

1. **Subí `ultimo_mes.zip`** al entorno de Colab: abrí el panel **Archivos** (ícono de carpeta 📁 a la izquierda) y arrastrá el `.zip` a `/content/`.
   > El zip pesa ~7 GB: la subida puede tardar. No cierres la pestaña mientras sube.
2. Ejecutá las celdas en orden (menú *Entorno de ejecución → Ejecutar todo*).
3. Al terminar, el notebook **descarga automáticamente** los dos archivos:
   - `MMAAAA_pais_parte1COMPLETO.csv.gz` (días 01–15)
   - `MMAAAA_pais_parte2COMPLETO.csv.gz` (días 16–último)
4. Después subí esos `.csv.gz` a tu `2026A.zip` en Drive y corré los notebooks 01 y 02.

> El precio del diario está en pesos y se exporta en **centavos** (×100), idéntico al formato oficial del SEPA. Los `id_sucursal` con ceros a la izquierda (`004`) se normalizan a `4` para que matcheen el maestro de sucursales.


In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
# Ruta del zip diario que subiste al entorno de Colab:
ZIP_DIARIO = '/content/ultimo_mes.zip'

# Mes a procesar: None = último mes presente en el zip. O forzar 'YYYY-MM'.
MES_FORZADO = None

# Dejar el precio en centavos (formato oficial del SEPA). No cambiar salvo que sepas lo que hacés.
PRECIO_EN_CENTAVOS = True

# Carpeta de salida dentro del entorno de Colab
OUTPUT_DIR = '/content/salida_consolidada'

# DEBUG (dejar en None para producción):
LIMITE_COMERCIOS = None   # procesar solo N comercios
LIMITE_DIAS = None        # procesar solo los primeros N días


In [ ]:
# ============================================================
# Imports y verificación del archivo subido
# ============================================================
import io, gc, re, sys, time, gzip, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

ZIP_DIARIO = Path(ZIP_DIARIO)
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_DIARIO.exists():
    raise FileNotFoundError(
        f'No encuentro {ZIP_DIARIO}.\n'
        'Subí ultimo_mes.zip al panel Archivos (carpeta 📁) en /content/ y volvé a ejecutar.'
    )
print(f'Zip diario OK: {ZIP_DIARIO}  ({ZIP_DIARIO.stat().st_size/1024**3:.2f} GB)')


In [ ]:
# ============================================================
# Motor de consolidación (funciones)
# ============================================================
_COLS_PROD = ["id_comercio", "id_bandera", "id_sucursal", "id_producto", "productos_precio_lista"]

_RE_NESTED = re.compile(r"comercio-sepa-(\d+)_", re.IGNORECASE)
_RE_DIA = re.compile(r"^(\d{4})-(\d{2})-(\d{2})$")


def _norm_suc(serie: pd.Series) -> pd.Series:
    """
    El diario rellena id_sucursal con ceros a la izquierda ('004'); el semestral
    oficial y el maestro de sucursales usan el id sin relleno ('4'). Hay que
    quitar los ceros para que el join geográfico de los notebooks matchee.
    """
    return serie.astype(str).str.strip().str.lstrip("0").replace("", "0")


# ============================================================================
# Funciones de lectura
# ============================================================================
def detectar_dias(zf: zipfile.ZipFile):
    """Devuelve (lista_dias 'YYYY-MM-DD' ordenada, anio, mes) presentes en el zip."""
    dias = set()
    for nombre in zf.namelist():
        top = nombre.split("/", 1)[0]
        if _RE_DIA.match(top):
            dias.add(top)
    if not dias:
        raise RuntimeError("No se encontraron carpetas YYYY-MM-DD en el zip diario.")
    dias = sorted(dias)
    if MES_FORZADO:
        dias = [d for d in dias if d.startswith(MES_FORZADO)]
        if not dias:
            raise RuntimeError(f"El mes forzado {MES_FORZADO} no está en el zip.")
    else:
        # Quedarnos con el mes más reciente presente
        ultimo_mes = dias[-1][:7]
        dias = [d for d in dias if d.startswith(ultimo_mes)]
    if LIMITE_DIAS:
        dias = dias[:LIMITE_DIAS]
    anio, mes = int(dias[0][:4]), int(dias[0][5:7])
    return dias, anio, mes


def mapear_comercios(zf: zipfile.ZipFile, dias):
    """comercio_id (str) -> {dia 'YYYY-MM-DD' -> nombre del zip anidado}."""
    setdias = set(dias)
    mapa = {}
    for nombre in zf.namelist():
        if not nombre.endswith(".zip"):
            continue
        top = nombre.split("/", 1)[0]
        if top not in setdias:
            continue
        m = _RE_NESTED.search(nombre)
        if not m:
            continue
        cid = m.group(1)
        mapa.setdefault(cid, {})[top] = nombre
    return mapa


def leer_nested(zf: zipfile.ZipFile, nested_name: str):
    """
    Lee un zip anidado de un comercio-día.
    Devuelve (df_precios, dict_provincia) donde:
      - df_precios: columnas [id_bandera, id_sucursal, id_producto, precio]
        deduplicado por (bandera, sucursal, producto), precio en pesos (float).
      - dict_provincia: (id_bandera, id_sucursal) -> sucursales_provincia
    Maneja zips vacíos/corruptos devolviendo (None, {}).
    """
    try:
        raw = zf.read(nested_name)
        if not raw:
            return None, {}
        inner = zipfile.ZipFile(io.BytesIO(raw))
    except Exception as e:
        print(f"      ⚠️  {Path(nested_name).name}: no se pudo abrir ({e})")
        return None, {}

    nombres = inner.namelist()

    # --- provincia por sucursal ---
    prov = {}
    if "sucursales.csv" in nombres:
        try:
            with inner.open("sucursales.csv") as f:
                ds = pd.read_csv(
                    io.TextIOWrapper(f, encoding="utf-8-sig", errors="replace"),
                    sep="|", dtype=str,
                    usecols=["id_bandera", "id_sucursal", "sucursales_provincia"],
                    on_bad_lines="skip",
                )
            ds = ds.dropna(subset=["id_sucursal"])
            ds["id_sucursal"] = _norm_suc(ds["id_sucursal"])
            prov = {
                (b, s): p
                for b, s, p in zip(ds["id_bandera"], ds["id_sucursal"], ds["sucursales_provincia"])
            }
        except Exception:
            prov = {}

    # --- precios ---
    if "productos.csv" not in nombres:
        return None, prov
    try:
        partes = []
        with inner.open("productos.csv") as f:
            tw = io.TextIOWrapper(f, encoding="utf-8-sig", errors="replace")
            for chunk in pd.read_csv(
                tw, sep="|", dtype=str, usecols=_COLS_PROD,
                chunksize=500_000, on_bad_lines="skip",
            ):
                partes.append(chunk)
        if not partes:
            return None, prov
        dp = pd.concat(partes, ignore_index=True)
    except Exception as e:
        print(f"      ⚠️  {Path(nested_name).name}: productos.csv ilegible ({e})")
        return None, prov

    dp["precio"] = pd.to_numeric(dp["productos_precio_lista"], errors="coerce")
    dp = dp[dp["precio"] > 0]
    dp["id_sucursal"] = _norm_suc(dp["id_sucursal"])
    dp = dp[["id_bandera", "id_sucursal", "id_producto", "precio"]]
    # Si un producto aparece más de una vez en el día para la misma sucursal,
    # quedarnos con el último (igual criterio que el SEPA al consolidar).
    dp = dp.drop_duplicates(subset=["id_bandera", "id_sucursal", "id_producto"], keep="last")
    return dp, prov


# ============================================================================
# Procesamiento por comercio (acota la RAM al comercio más grande)
# ============================================================================
def procesar_comercio(zf, cid, dias_nested, columnas_dia):
    """
    Construye el frame wide de un comercio.
    columnas_dia: lista de ('YYYY-MM-DD', 'precio_YYYYMMDD') de TODO el mes.
    Devuelve un DataFrame con columnas:
        id_comercio,id_bandera,id_sucursal,sucursales_provincia,id_producto,
        precio_YYYYMMDD...   (precio en centavos enteros, NaN donde falta)
    """
    factor = 100 if PRECIO_EN_CENTAVOS else 1
    wide = None
    prov_global = {}

    for dia, col in columnas_dia:
        nested = dias_nested.get(dia)
        if nested is None:
            continue
        dp, prov = leer_nested(zf, nested)
        if prov:
            prov_global.update(prov)
        if dp is None or len(dp) == 0:
            continue
        s = dp.set_index(["id_bandera", "id_sucursal", "id_producto"])["precio"]
        s = (s * factor).round().astype("float32")
        s.name = col
        if wide is None:
            wide = s.to_frame()
        else:
            wide = wide.join(s, how="outer")
        del dp, s
        gc.collect()

    if wide is None:
        return None

    wide = wide.reset_index()
    wide.insert(0, "id_comercio", cid)
    # provincia
    claves = list(zip(wide["id_bandera"], wide["id_sucursal"]))
    wide["sucursales_provincia"] = [prov_global.get(k, "") for k in claves]

    # Orden final de columnas
    cols_precio = [c for _, c in columnas_dia]
    # asegurar que existan todas las columnas de día (NaN si el comercio no
    # reportó ese día)
    for c in cols_precio:
        if c not in wide.columns:
            wide[c] = np.nan
    orden = ["id_comercio", "id_bandera", "id_sucursal", "sucursales_provincia",
             "id_producto"] + cols_precio
    return wide[orden]


def escribir_parte(handle, df, cols_precio_parte):
    """Escribe las filas de un comercio en el .csv.gz de una parte (sin header)."""
    base = ["id_comercio", "id_bandera", "id_sucursal", "sucursales_provincia", "id_producto"]
    sub = df[base + cols_precio_parte].copy()
    # Precios -> enteros nullables para imprimir sin '.0' y con 'NA' en faltantes
    for c in cols_precio_parte:
        sub[c] = sub[c].round().astype("Int64")
    sub.to_csv(handle, header=False, index=False, na_rep="NA", lineterminator="\n")


In [ ]:
# ============================================================
# EJECUTAR — consolida y descarga los dos .csv.gz
# ============================================================
t0 = time.time()
zf = zipfile.ZipFile(ZIP_DIARIO, 'r')
dias, anio, mes = detectar_dias(zf)
mmaaaa = f'{mes:02d}{anio}'
print(f'Mes detectado: {anio}-{mes:02d}  |  {len(dias)} días: {dias[0]} … {dias[-1]}')

columnas_dia = [(d, f"precio_{d.replace('-', '')}") for d in dias]
dias_n = [int(d[8:10]) for d in dias]
cols_p1 = [c for (d, c), dn in zip(columnas_dia, dias_n) if dn <= 15]
cols_p2 = [c for (d, c), dn in zip(columnas_dia, dias_n) if dn >= 16]

mapa = mapear_comercios(zf, dias)
comercios = sorted(mapa.keys(), key=lambda x: int(x) if x.isdigit() else 0)
if LIMITE_COMERCIOS:
    comercios = comercios[:LIMITE_COMERCIOS]
print(f'Comercios a procesar: {len(comercios)} -> {comercios}')

f1 = OUTPUT_DIR / f'{mmaaaa}_pais_parte1COMPLETO.csv.gz'
f2 = OUTPUT_DIR / f'{mmaaaa}_pais_parte2COMPLETO.csv.gz'
base_cols = ['id_comercio', 'id_bandera', 'id_sucursal', 'sucursales_provincia', 'id_producto']

h1 = gzip.open(f1, 'wt', encoding='utf-8', newline='')
h1.write(','.join(base_cols + cols_p1) + '\n')
h2 = None
if cols_p2:
    h2 = gzip.open(f2, 'wt', encoding='utf-8', newline='')
    h2.write(','.join(base_cols + cols_p2) + '\n')

tot_filas = 0
eans = set()
for i, cid in enumerate(comercios, 1):
    tc = time.time()
    df = procesar_comercio(zf, cid, mapa[cid], columnas_dia)
    if df is None or len(df) == 0:
        print(f'  [{i}/{len(comercios)}] comercio {cid}: sin datos')
        continue
    escribir_parte(h1, df, cols_p1)
    if h2 is not None:
        escribir_parte(h2, df, cols_p2)
    tot_filas += len(df)
    eans.update(df['id_producto'].unique())
    print(f'  [{i}/{len(comercios)}] comercio {cid}: {len(df):,} filas ({time.time()-tc:.1f}s)')
    del df; gc.collect()

h1.close()
if h2 is not None:
    h2.close()
zf.close()

print('-' * 70)
print(f'Filas totales (producto×sucursal): {tot_filas:,}')
print(f'Productos (EAN) únicos:            {len(eans):,}')
print(f'  {f1.name}: {f1.stat().st_size/1024**2:.1f} MB  ({len(cols_p1)} días)')
if h2 is not None:
    print(f'  {f2.name}: {f2.stat().st_size/1024**2:.1f} MB  ({len(cols_p2)} días)')
else:
    print('  (sin parte2: el mes aún no tiene días >= 16)')
print(f'Listo en {(time.time()-t0)/60:.1f} min.')

# ── Descargar los archivos ──────────────────────────────────
from google.colab import files
print('\nDescargando…')
files.download(str(f1))
if h2 is not None:
    files.download(str(f2))
